# Treinamento de Modelo Qwen com Unsloth

Este notebook treina um modelo de linguagem Qwen 7B usando quantização 4-bit e fine-tuning eficiente com Unsloth.

## Características:
- Modelo: Qwen2.5-7B-Instruct (4-bit quantizado)
- Framework: Unsloth + TRL (Transformer Reinforcement Learning)
- Dados: Treinamento com mensagens em formato chat
- Otimização: LoRA (Low-Rank Adaptation)
- Saída: Modelo em formato GGUF

## 1. Instalar Dependências

In [ ]:
# Instalar os pacotes necessários
!pip install -q accelerate bitsandbytes datasets peft torch torchaudio torchvision trl unsloth

## 2. Configurar Variáveis de Ambiente

In [ ]:
import os

# Configurar para usar segmentos expandíveis de CUDA (melhor para GPUs com memória limitada)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("✓ Variáveis de ambiente configuradas")

## 3. Carregar Imports Necessários

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json
import torch

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA disponível: {torch.cuda.is_available()}")

## 4. Carregar o Modelo Base

**Nota sobre modelos alternativos:**
- Para GPUs com menos memória: `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit`
- Padrão: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`

In [ ]:
# Carregar modelo pré-treinado com quantização 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=512,
    load_in_4bit=True,
)

print("✓ Modelo carregado com sucesso")

## 5. Aplicar LoRA (Low-Rank Adaptation)

LoRA reduz drasticamente o número de parâmetros treináveis, tornando o treinamento mais eficiente.

In [ ]:
# Adicionar adaptadores LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank da matriz low-rank
    target_modules=["q_proj", "v_proj"],  # Módulos a serem adaptados
)

print("✓ Adaptadores LoRA aplicados com sucesso")

## 6. Carregar Dados de Treinamento

Os dados devem estar em formato JSONL com campo 'messages' contendo histórico de chat.

In [ ]:
# Carregar dados do arquivo JSONL
# Exemplo esperado: {"messages": [{"role": "...", "content": "..."}]}

# Se você está no Kaggle, você pode fazer upload do arquivo treino.jsonl
# Para este exemplo, vamos usar um arquivo local

# Verifique o caminho do arquivo de treino
train_file_path = "treino.jsonl"  # Altere se necessário

if not os.path.exists(train_file_path):
    print("⚠️ Arquivo treino.jsonl não encontrado!")
    print("Por favor, faça upload do arquivo ou ajuste o caminho.")
else:
    # Carregar dados
    records = [json.loads(line) for line in open(train_file_path) if line.strip()]
    print(f"✓ {len(records)} exemplos de treinamento carregados")
    
    # Mostrar exemplo
    if records:
        print(f"\nExemplo de dado:")
        print(json.dumps(records[0], indent=2, ensure_ascii=False)[:500])

## 7. Preparar Dataset

In [ ]:
# Aplicar template de chat e criar campo 'text'
for record in records:
    record["text"] = tokenizer.apply_chat_template(
        record["messages"], 
        tokenize=False
    )

# Criar dataset
dataset = Dataset.from_list(records)

print(f"✓ Dataset preparado com {len(dataset)} exemplos")
print(f"\nExemplo de texto tokenizado (primeiros 200 caracteres):")
print(dataset[0]["text"][:200])

## 8. Configurar e Executar Treinamento

**Dicas para otimizar o treinamento:**
- `per_device_train_batch_size`: Reduzir se houver erro de memória
- `gradient_accumulation_steps`: Aumentar para simular batch maior
- `num_train_epochs`: Aumentar para dataset pequeno (ex: 20-25)
- `bf16`: Usar precisão bfloat16 (requer GPU moderna)

In [ ]:
# Configurar trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./outputs",
        per_device_train_batch_size=1,  # Batch size por GPU
        gradient_accumulation_steps=4,  # Acumular gradientes
        gradient_checkpointing=True,    # Economizar memória
        bf16=True,                      # Precisão bfloat16
        max_length=512,                 # Comprimento máximo de sequência
        dataset_text_field="text",     # Campo com dados de treino
        num_train_epochs=25,            # Número de épocas (aumentado para dataset pequeno)
        save_steps=100,                 # Salvar checkpoint a cada 100 steps
        logging_steps=10,               # Log a cada 10 steps
        weight_decay=0.01,              # Regularização
    ),
)

print("✓ Trainer configurado com sucesso")

## 9. Iniciar Treinamento

⏱️ **Tempo estimado de treinamento:** Depende do número de exemplos e da GPU disponível

In [ ]:
# Executar treinamento
print("🚀 Iniciando treinamento...\n")
trainer.train()
print("\n✓ Treinamento concluído!")

## 10. Salvar Modelo em Formato GGUF

O formato GGUF é otimizado para inferência com CPU e é compatível com Ollama e outras ferramentas.

In [ ]:
# Salvar modelo em formato GGUF
print("💾 Salvando modelo em formato GGUF...\n")
model.save_pretrained_gguf(
    "outputs/gguf",
    tokenizer,
    quantization_method="q4_k_m"  # Quantização 4-bit
)
print("✓ Modelo salvo em outputs/gguf/")

## 11. Próximos Passos

### Opção 1: Usar com Ollama (Local)
```bash
sed -i "s|FROM Qwen2.5-7B-Instruct.Q4_K_M.gguf|FROM $(pwd)/outputs/gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf|" outputs/gguf/Modelfile
ollama create meu-modelo -f outputs/gguf/Modelfile
ollama run meu-modelo
```

### Opção 2: Fazer Inferência Direto
```python
from llama_cpp import Llama

llm = Llama(model_path="outputs/gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf", n_gpu_layers=-1)
response = llm("Seu prompt aqui")
```

### Opção 3: Fazer Download dos Arquivos
Se você está no Kaggle, faça download dos arquivos da pasta `outputs/gguf/`

## 12. Verificar Arquivos Gerados

In [ ]:
import os

if os.path.exists("outputs/gguf"):
    print("📁 Arquivos gerados em outputs/gguf/:")
    for file in os.listdir("outputs/gguf"):
        file_path = os.path.join("outputs/gguf", file)
        file_size = os.path.getsize(file_path) / (1024**3)  # Converter para GB
        print(f"  - {file} ({file_size:.2f} GB)")
else:
    print("❌ Pasta outputs/gguf não encontrada")